In [ ]:
import os
import numpy as np
import xarray as xr
import zarr
from numcodecs import Blosc
from datetime import datetime


start_year = 1961
end_year = 2024
years = list(range(start_year, end_year + 1))

nc_dir = "data/netcdf/"  

zarr_path = "SPARTACUS.zarr"
fill_value = -999
scale_factor = 0.1


store = zarr.storage.LocalStore(zarr_path)
root = zarr.group(store=store, overwrite=True)
compressor = zarr.codecs.BloscCodec()


x_extent = np.arange(112500, 695501, 1000)  
y_extent = np.arange(258500, 586501, 1000)  

def is_leap_year(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

days_per_year = [366 if is_leap_year(y) else 365 for y in years]
total_days = sum(days_per_year)

root.create_array(
    name="time",
    shape=(total_days,),
    dtype="int32",
    chunks=(total_days,),
    dimension_names=["time"],
    attributes={
        "units": "days since 1961-01-01",
        "calendar": "proleptic_gregorian"
    },
    overwrite=True
)[:] = np.arange(total_days, dtype="int32")


x_array = root.create_array(
    name="x",
    shape=x_extent.shape,
    dtype="int32",
    chunks=(len(x_extent),),
    dimension_names=["x"],
    overwrite=True
)
x_array[:] = x_extent

y_array = root.create_array(
    name="y",
    shape=y_extent.shape,
    dtype="int32",
    chunks=(len(y_extent),),
    dimension_names=["y"],
    overwrite=True
)
y_array[:] = y_extent

print(total_days)
print(x_extent.shape)


23376
(584,)


In [15]:
zarr.consolidate_metadata(store)

/home/katharina/miniconda3/envs/env_zarr/lib/python3.11/site-packages/zarr/api/asynchronous.py:213: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


<Group file://SPARTACUS.zarr>

In [16]:
shape = (total_days, len(y_extent), len(x_extent))
chunk_shape = (365, len(y_extent), len(x_extent))

tx_array = root.create_array(
    name="TX",
    shape=shape,
    chunks=chunk_shape,
    dtype="int16",
    fill_value=fill_value,
    compressor=compressor,
    dimension_names=["time", "y", "x"],
    attributes={
        "units": "degree_Celsius",
        "scale_factor": 0.1,
        "description": (
            "daily maximum (derived from daily maxima measured between "
            "19:00 CET of the previous day and 19:00 CET of the respective day)"
        ),
        "_FillValue": fill_value,
        "missing_value": -999,
        "cell_method": "time: maximum (19:00 CET day-1 to 19:00 CET)",
        "standard_name": "surface_temperature",
        "long_name": "daily maximum of air temperature",
        "grid_mapping": "lambert_conformal_conic",
    },
    overwrite=True
)



/home/katharina/miniconda3/envs/env_zarr/lib/python3.11/site-packages/zarr/core/group.py:2481: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


In [17]:
ds_nc = xr.open_dataset("data/netcdf/SPARTACUS2-DAILY_TX_1999.nc", mask_and_scale=False)
ds_nc["TX"]

<xarray.DataArray 'TX' (time: 365, y: 329, x: 584)> Size: 281MB
[70129640 values with dtype=int32]
Coordinates:
    lambert_conformal_conic  float64 8B ...
    lat                      (y, x) float32 769kB ...
    lon                      (y, x) float32 769kB ...
  * time                     (time) datetime64[ns] 3kB 1999-01-01 ... 1999-12-31
  * x                        (x) int32 2kB 112500 113500 ... 694500 695500
  * y                        (y) int32 1kB 258500 259500 ... 585500 586500
Attributes:
    _FillValue:      -999
    cell_method:     time: maximum (19:00 CET day-1 to 19:00 CET)
    description:     daily maximum (derived from daily maxima measured betwee...
    esri_pe_string:  PROJCS["ETRS89 / Austria Lambert",GEOGCS["ETRS89",DATUM[...
    grid_mapping:    lambert_conformal_conic
    long_name:       daily maximum of air temperature
    standard_name:   surface_temperature
    units:           degree_Celsius
    scale_factor:    0.1
    missing_value:   -999

In [18]:
current_start = 0

for year in years:
    nc_path = os.path.join(nc_dir, f"SPARTACUS2-DAILY_TX_{year}.nc")
    if not os.path.exists(nc_path):
        print(f"Datei fehlt: {nc_path}")
        continue

    ds = xr.open_dataset(nc_path, mask_and_scale=False)
    
    tx = ds["TX"].values.astype("float32")

    num_days = tx.shape[0]
    current_end = current_start + num_days

    tx_array[current_start:current_end, :, :] = tx
    #print(f"{year} eingefügt ({num_days} Tage)")

    current_start = current_end

print("Zarr-Konvertierung abgeschlossen.")


Zarr-Konvertierung abgeschlossen.


In [19]:
ds_nc = xr.open_dataset("data/TN_data/SPARTACUS2-DAILY_TN_1999.nc", mask_and_scale=False)
ds_nc["TN"]

<xarray.DataArray 'TN' (time: 365, y: 329, x: 584)> Size: 281MB
[70129640 values with dtype=int32]
Coordinates:
    lambert_conformal_conic  float64 8B ...
    lat                      (y, x) float32 769kB ...
    lon                      (y, x) float32 769kB ...
  * time                     (time) datetime64[ns] 3kB 1999-01-01 ... 1999-12-31
  * x                        (x) int32 2kB 112500 113500 ... 694500 695500
  * y                        (y) int32 1kB 258500 259500 ... 585500 586500
Attributes:
    _FillValue:      -999
    cell_method:     time: minimum (19:00 CET day-1 to 19:00 CET)
    description:     daily minimum (derived from daily minima measured betwee...
    esri_pe_string:  PROJCS["ETRS89 / Austria Lambert",GEOGCS["ETRS89",DATUM[...
    grid_mapping:    lambert_conformal_conic
    long_name:       daily minimum of air temperature
    standard_name:   surface_temperature
    units:           degree_Celsius
    scale_factor:    0.1
    missing_value:   -999

In [20]:
nc_dir = 'data/TN_data'

tn_array = root.create_array(
    name="TN",
    shape=shape,
    chunks=chunk_shape,
    dtype="int16",
    fill_value=fill_value,
    compressor=compressor,
    dimension_names=["time", "y", "x"],
    attributes={
        "units": "degree_Celsius",
        "scale_factor": 0.1,
        "description": (
            "daily minimum (derived from daily minima measured between "
            "19:00 CET of the previous day and 19:00 CET of the respective day)"
        ),
        "_FillValue": fill_value,
        "missing_value": -999,
        "cell_method": "time: minimum (19:00 CET day-1 to 19:00 CET)",
        "standard_name": "surface_temperature",
        "long_name": "daily minimum of air temperature",
        "grid_mapping": "lambert_conformal_conic",
    },
    overwrite=True
)


/home/katharina/miniconda3/envs/env_zarr/lib/python3.11/site-packages/zarr/core/group.py:2481: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


In [21]:

current_start = 0

for year in years:
    nc_path = os.path.join(nc_dir, f"SPARTACUS2-DAILY_TN_{year}.nc")
    if not os.path.exists(nc_path):
        print(f"Datei fehlt: {nc_path}")
        continue

    ds = xr.open_dataset(nc_path, mask_and_scale=False)
    
    tn = ds["TN"].values.astype("float32")

    num_days = tn.shape[0]
    current_end = current_start + num_days

    tn_array[current_start:current_end, :, :] = tn
    #print(f"{year} eingefügt ({num_days} Tage)")

    current_start = current_end

print("Zarr-Konvertierung abgeschlossen.")


Zarr-Konvertierung abgeschlossen.


In [22]:
ds_nc = xr.open_dataset("data/SA_data/SPARTACUS2-DAILY_SA_1999.nc", mask_and_scale=False)
ds_nc.load()


<xarray.Dataset> Size: 282MB
Dimensions:                  (time: 365, y: 329, x: 584)
Coordinates:
    lambert_conformal_conic  float64 8B nan
    lat                      (y, x) float32 769kB 46.16 46.17 ... 49.11 49.11
    lon                      (y, x) float32 769kB 9.609 9.622 ... 17.37 17.38
  * time                     (time) datetime64[ns] 3kB 1999-01-01 ... 1999-12-31
  * x                        (x) int32 2kB 112500 113500 ... 694500 695500
  * y                        (y) int32 1kB 258500 259500 ... 585500 586500
Data variables:
    SA                       (time, y, x) int32 281MB -999 -999 ... -999 -999
Attributes: (12/14)
    Conventions:         CF-1.7
    author:              GeoSphere Austria (kontakt@geosphere.at)
    comment:             No additional comments
    crs:                 EPSG:3416
    freq:                1D
    grid_mapping:        lambert_conformal_conic
    ...                  ...
    name:                spartacus-daily v2.1
    references:          temperature - doi:10.1007/s00704-015-1411-4, precipi...
    source:              geostatistical interpolation of surface station obse...
    spatial_domain:      SPARTACUS
    spatial_resolution:  1000
    title:               SPARTACUS - Spatial Dataset for Climate in Austria

In [23]:
nc_dir = 'data/SA_data'

sa_array = root.create_array(
    name="SA",
    shape=shape,
    chunks=chunk_shape,
    dtype="int16",
    fill_value=fill_value,
    compressor=compressor,
    dimension_names=["time", "y", "x"],
    attributes={
        "_FillValue": -999,
        "missing_value": -999,
        "scale_factor": 360.0,
        "units": "s",
        "standard_name": "duration_of_sunshine",
        "long_name": "daily duration of sunshine",
        "description": (
            "daily sum (derived from daily sums measured between "
            "00:00 CET and 24:00 CET of the respective day)"
        ),
        "cell_method": "time: sum (0:00 CET to 24:00 CET)",
        "grid_mapping": "lambert_conformal_conic",
        "esri_pe_string": 'PROJCS["ETRS89 / Austria Lambert",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.01745329251994328,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],UNIT["metre",1,AUTHORITY["EPSG","9001"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["standard_parallel_1",49],PARAMETER["standard_parallel_2",46],PARAMETER["latitude_of_origin",47.5],PARAMETER["central_meridian",13.33333333333333],PARAMETER["false_easting",400000],PARAMETER["false_northing",400000],AUTHORITY["EPSG","3416"],AXIS["Y",EAST],AXIS["X",NORTH]]',
    },
    overwrite=True
) 

current_start = 0

for year in years:
    nc_path = os.path.join(nc_dir, f"SPARTACUS2-DAILY_SA_{year}.nc")
    if not os.path.exists(nc_path):
        print(f"Datei fehlt: {nc_path}")
        continue

    ds = xr.open_dataset(nc_path, mask_and_scale=False)

    sa = ds["SA"].values.astype("float32")  # Rohdaten
    sa[sa == -999] = np.nan  # oder -9999, je nach _FillValue im Original!

    sa_filled = np.where(np.isnan(sa), fill_value, sa).astype("int16")  # falls du speichern willst wie bei TX/TN

    num_days = sa.shape[0]
    current_end = current_start + num_days

    sa_array[current_start:current_end, :, :] = sa_filled  # ⬅️ falls du "sa_array" angelegt hast
    #print(f"{year} eingefügt ({num_days} Tage)")

    current_start = current_end

print("Zarr-Konvertierung abgeschlossen.")



/home/katharina/miniconda3/envs/env_zarr/lib/python3.11/site-packages/zarr/core/group.py:2481: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


Zarr-Konvertierung abgeschlossen.


In [24]:
ds_nc = xr.open_dataset("data/RR_data/SPARTACUS2-DAILY_RR_1999.nc", mask_and_scale=False)
ds_nc.load()

<xarray.Dataset> Size: 282MB
Dimensions:                  (time: 365, y: 329, x: 584)
Coordinates:
    lambert_conformal_conic  float64 8B nan
    lat                      (y, x) float32 769kB 46.16 46.17 ... 49.11 49.11
    lon                      (y, x) float32 769kB 9.609 9.622 ... 17.37 17.38
  * time                     (time) datetime64[ns] 3kB 1999-01-01 ... 1999-12-31
  * x                        (x) int32 2kB 112500 113500 ... 694500 695500
  * y                        (y) int32 1kB 258500 259500 ... 585500 586500
Data variables:
    RR                       (time, y, x) int32 281MB -999 -999 ... -999 -999
Attributes: (12/14)
    Conventions:         CF-1.7
    author:              GeoSphere Austria (kontakt@geosphere.at)
    comment:             No additional comments
    crs:                 EPSG:3416
    freq:                1D
    grid_mapping:        lambert_conformal_conic
    ...                  ...
    name:                spartacus-daily v2.1
    references:          temperature - doi:10.1007/s00704-015-1411-4, precipi...
    source:              geostatistical interpolation of surface station obse...
    spatial_domain:      SPARTACUS
    spatial_resolution:  1000
    title:               SPARTACUS - Spatial Dataset for Climate in Austria

In [25]:
nc_dir = 'data/RR_data/' 

rr_array = root.create_array(
    name="RR",  
    shape=shape,
    chunks=chunk_shape,
    dtype="int16",  
    fill_value=fill_value,
    compressor=compressor,
    dimension_names=["time", "y", "x"],
    attributes={
        "units": "kg m-2",
        "scale_factor": 0.1,
        "_FillValue": fill_value,
        "missing_value": -999,
        "description": "daily sum (derived from daily sums measured by climatological stations only between 07:00 CET of the respective day and 07:00 CET of the following day)",
        "cell_method": "time: sum (7:00 CET to 7:00 CET day+1)",
        "long_name": "daily precipitation sum",
        "standard_name": "precipitation_amount",
        "grid_mapping": "lambert_conformal_conic",
        "esri_pe_string": 'PROJCS["ETRS89 / Austria Lambert",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.01745329251994328,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],UNIT["metre",1,AUTHORITY["EPSG","9001"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["standard_parallel_1",49],PARAMETER["standard_parallel_2",46],PARAMETER["latitude_of_origin",47.5],PARAMETER["central_meridian",13.33333333333333],PARAMETER["false_easting",400000],PARAMETER["false_northing",400000],AUTHORITY["EPSG","3416"],AXIS["Y",EAST],AXIS["X",NORTH]]'
    },
    overwrite=True
)

current_start = 0

for year in years:
    nc_path = os.path.join(nc_dir, f"SPARTACUS2-DAILY_RR_{year}.nc")
    if not os.path.exists(nc_path):
        print(f"Datei fehlt: {nc_path}")
        continue

    ds = xr.open_dataset(nc_path, mask_and_scale=False)

    rr = ds["RR"].values.astype("float32")  # Rohdaten lesen
    rr[rr == -999] = np.nan  # Achtung: manchmal -9999 – prüfen im Original!

    rr_filled = np.where(np.isnan(rr), fill_value, rr).astype("int16")  # Optional skalieren, falls nötig

    num_days = rr.shape[0]
    current_end = current_start + num_days

    rr_array[current_start:current_end, :, :] = rr_filled  # ⬅️ rr_array muss vorher per create_array erstellt worden sein
    #print(f"{year} eingefügt ({num_days} Tage)")

    current_start = current_end

print("Zarr-Konvertierung abgeschlossen.")



/home/katharina/miniconda3/envs/env_zarr/lib/python3.11/site-packages/zarr/core/group.py:2481: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(


Zarr-Konvertierung abgeschlossen.


In [26]:
# TESTING TX


import numpy as np
import xarray as xr

# Zarr-Datei laden
ds = xr.open_zarr("SPARTACUS.zarr")

# TX-Werte am 1. Januar 1999 extrahieren
tx_day = ds["TX"].isel(time=13879).values

# Gültige Werte anzeigen
valid_values = tx_day[~np.isnan(tx_day)]

print(f"📅 Anzahl gültiger Werte am 1999-01-01: {valid_values.size}")
print(f"🔍 Erste 10 gültige Werte: {valid_values[:10]}")

print("Zarr-Wert [0, 0, 124]:", ds["TX"].isel(time=13879, y=0, x=124).values)

ds_nc = xr.open_dataset("data/netcdf/SPARTACUS2-DAILY_TX_1999.nc")

tx_nc_day0 = ds_nc["TX"].isel(time=0).values
print("NetCDF-Wert [0, 0, 124]:", tx_nc_day0[0, 124])

# Tag 0 (1999-01-01)
tx_nc_day0 = ds_nc["TX"].isel(time=0).values

# Gültige Werte (nicht NaN)
valid_nc = tx_nc_day0[~np.isnan(tx_nc_day0)]

print(f"📄 NetCDF gültige Werte: {valid_nc[:10]}")




KeyError: "No variable named 'TX'. Variables on the dataset include ['time', 'x', 'y']"

In [ ]:
# Testing TN
import numpy as np
import xarray as xr

# Zarr-Datei laden
ds = xr.open_zarr("SPARTACUS.zarr")

# TX-Werte am 1. Januar 1999 extrahieren
tx_day = ds["TN"].isel(time=13879).values

# Gültige Werte anzeigen
valid_values = tx_day[~np.isnan(tx_day)]

print(f"Anzahl gültiger Werte am 1999-01-01: {valid_values.size}")
print(f"Erste 10 gültige Werte: {valid_values[:10]}")

print("Zarr-Wert [0, 0, 124]:", ds["TX"].isel(time=13879, y=0, x=124).values)

ds_nc = xr.open_dataset("data/TN_data/SPARTACUS2-DAILY_TN_1999.nc")

tx_nc_day0 = ds_nc["TN"].isel(time=0).values
print("NetCDF-Wert [0, 0, 124]:", tx_nc_day0[0, 124])

# Tag 0 (1999-01-01)
tx_nc_day0 = ds_nc["TN"].isel(time=0).values

# Gültige Werte (nicht NaN)
valid_nc = tx_nc_day0[~np.isnan(tx_nc_day0)]

print(f"NetCDF gültige Werte: {valid_nc[:10]}")



Anzahl gültiger Werte am 1999-01-01: 105736
Erste 10 gültige Werte: [-3.2 -4.3 -3.  -3.8 -4.2 -4.2 -3.3 -4.5 -0.4 -3.7]
Zarr-Wert [0, 0, 124]: 2.6
NetCDF-Wert [0, 0, 124]: -3.2
NetCDF gültige Werte: [-3.2 -4.3 -3.  -3.8 -4.2 -4.2 -3.3 -4.5 -0.4 -3.7]


In [ ]:
# Testing SA
import numpy as np
import xarray as xr

# Parameter
zarr_path = "SPARTACUS.zarr"
nc_path = "data/SA_data/SPARTACUS2-DAILY_SA_1999.nc"
fill_value = -999

# Zarr öffnen
ds_zarr = xr.open_zarr(zarr_path)

target_date = np.datetime64("1999-06-01")
time_index = int((target_date - np.datetime64("1961-01-01")).astype("int32"))

# Zarr
sa_zarr = ds_zarr["SA"].isel(time=time_index).values.astype("float32")
sa_zarr[sa_zarr == -999] = np.nan

# NetCDF
ds_nc = xr.open_dataset(nc_path)
sa_nc = ds_nc["SA"].isel(time=151).values.astype("float32")  # 1999-06-01 = 151. Tag
sa_nc[sa_nc == -999] = np.nan

print("Zarr erste Werte:", sa_zarr[~np.isnan(sa_zarr)][:10])
print("NetCDF erste Werte:", sa_nc[~np.isnan(sa_nc)][:10])



Zarr erste Werte: [37440. 42480. 38160. 41400. 39600. 40320. 47520. 43560. 30960. 31680.]
NetCDF erste Werte: [37440. 42480. 38160. 41400. 39600. 40320. 47520. 43560. 30960. 31680.]


In [ ]:
# TESTING RR

import numpy as np
import xarray as xr

# --- Pfade ---
zarr_path = "SPARTACUS.zarr"
nc_path = "data/RR_data/SPARTACUS2-DAILY_RR_1999.nc"

fill_value = -999

# Zarr öffnen
ds_zarr = xr.open_zarr(zarr_path, consolidated=True)

target_date = np.datetime64("1999-06-01")
time_index = int((target_date - np.datetime64("1961-01-01")).astype("int32"))

# Zarr
rr_zarr = ds_zarr["RR"].isel(time=time_index).values.astype("float32")
rr_zarr[rr_zarr == -999] = np.nan

# NetCDF
ds_nc = xr.open_dataset(nc_path)
rr_nc = ds_nc["RR"].isel(time=151).values.astype("float32")  
rr_nc[rr_nc == -999] = np.nan

print("Zarr erste Werte:", rr_zarr[~np.isnan(rr_zarr)][:10])
print("NetCDF erste Werte:", rr_nc[~np.isnan(rr_nc)][:10])


Zarr erste Werte: [0.1 0.1 0.1 0.1 0.2 0.2 0.2 0.2 0.2 0.2]
NetCDF erste Werte: [0.1 0.1 0.1 0.1 0.2 0.2 0.2 0.2 0.2 0.2]


/tmp/ipykernel_517365/3864018235.py:1: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  xr.open_zarr(zarr_path)['TX']


KeyError: "No variable named 'TX'. Variables on the dataset include ['x', 'y', 'time']"